# D08 종목 기본정보 API 검증

## 목적
금융위원회 주식발행정보 API의 종목기본정보 기능을 이용하여
국내 상장 종목의 기본정보를 조회할 수 있는지 검증한다.

### 테스트 대상
- 기업명: 삼성전자
- 종목코드: 005930
- ISIN: KR7005930003
- 시장: KOSPI

### 검증 항목
1. 삼성전자 종목 기본정보 조회 가능 여부
2. ISIN 및 법인 식별정보 확인
3. 발행주식수 확인
4. 액면가 확인
5. 상장일 등 기본정보 확인
6. D06 주식시세정보와 연결 가능한 식별값 확인

> D08 API는 005930과 같은 6자리 단축 종목코드를 직접 조회키로 사용하는 대신
> ISIN, 법인등록번호, 기준일자 등을 이용한다.

In [ ]:
import requests
import pandas as pd
from getpass import getpass

pd.set_option("display.max_columns", None)

In [ ]:
# 인증키는 Notebook에 직접 저장하지 않고 실행 시 입력
SERVICE_KEY = getpass("공공데이터포털 Decoding 인증키 입력: ")

BASE_URL = (
    "https://apis.data.go.kr/1160100/"
    "GetStocIssuInfoService_V3/getItemBasiInfo_V3"
)

# 삼성전자
STOCK_CODE = "005930"
STOCK_NAME = "삼성전자"
ISIN_CODE = "KR7005930003"

# 과도한 조회로 인한 Timeout을 피하기 위해 특정 기준일 사용
BASE_DATE = "20261008"

In [ ]:
def get_stock_basic_info(
    service_key,
    isin_code,
    base_date,
    num_rows=10
):
    """
    금융위원회 종목기본정보 API 호출

    Parameters
    ----------
    service_key : str
        공공데이터포털 인증키
    isin_code : str
        ISIN 코드
    base_date : str
        기준일자 (YYYYMMDD)
    num_rows : int
        조회 건수

    Returns
    -------
    pandas.DataFrame
    """

    params = {
        "serviceKey": service_key,
        "pageNo": 1,
        "numOfRows": num_rows,
        "resultType": "json",
        "isinCd": isin_code,
        "basDt": base_date
    }

    try:
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=(5, 90)
        )

        print("HTTP 상태코드:", response.status_code)

        if response.status_code != 200:
            print("API 호출 실패")
            print(response.text[:1000])
            return pd.DataFrame()

        data = response.json()

        if "response" not in data:
            print("정상적인 API 응답 구조가 아닙니다.")
            print(data)
            return pd.DataFrame()

        header = data["response"]["header"]

        print(
            "API 결과:",
            header.get("resultCode"),
            header.get("resultMsg")
        )

        body = data["response"].get("body", {})
        items = body.get("items", {})

        if not items:
            print("조회 결과가 없습니다.")
            return pd.DataFrame()

        rows = items.get("item", [])

        if isinstance(rows, dict):
            rows = [rows]

        return pd.DataFrame(rows)

    except requests.exceptions.ReadTimeout:
        print("서버 응답시간 초과(ReadTimeout)")
        return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print("API 요청 오류:", e)
        return pd.DataFrame()

## 1. 삼성전자 종목 기본정보 조회

ISIN `KR7005930003`과 기준일자를 함께 지정하여 조회 범위를 좁힌다.

이전 테스트에서 ISIN만 사용했을 경우 서버 응답시간 초과가 발생했기 때문에
기준일자를 함께 지정하여 조회한다.

In [ ]:
df = get_stock_basic_info(
    service_key=SERVICE_KEY,
    isin_code=ISIN_CODE,
    base_date=BASE_DATE
)

print("조회 건수:", len(df))

if not df.empty:
    print("\n전체 컬럼:")
    print(df.columns.tolist())

    display(df)

In [ ]:
wanted_cols = [
    "basDt",           # 기준일자
    "crno",            # 법인등록번호
    "isinCd",          # ISIN
    "stckIssuCmpyNm",  # 주식발행회사명
    "isinCdNm",        # ISIN 명칭
    "scrsItmsKcd",     # 종목종류코드
    "scrsItmsKcdNm",   # 종목종류명
    "stckParPrc",      # 액면가
    "issuStckCnt",     # 발행주식수
    "lstgDt",          # 상장일
    "lstgAbolDt"       # 상장폐지일
]

existing_cols = [
    col for col in wanted_cols
    if col in df.columns
]

result = df[existing_cols].copy()

display(result)

## 2. 핵심 식별정보 검증

D08의 목적은 단순히 데이터를 조회하는 것뿐만 아니라
다른 데이터 소스와 종목을 정확히 연결할 수 있는지 확인하는 것이다.

예상 연결 구조:

D06 주식시세정보  
`005930`
↓  
삼성전자
↓  
ISIN `KR7005930003`
↓  
D08 종목기본정보

In [ ]:
print("===== D08 검증 결과 =====")

print("데이터 존재:", not df.empty)

if not df.empty:

    if "isinCd" in df.columns:
        print(
            "ISIN 일치:",
            (df["isinCd"] == ISIN_CODE).any()
        )

    if "stckIssuCmpyNm" in df.columns:
        print(
            "발행회사:",
            df["stckIssuCmpyNm"].iloc[0]
        )

    if "issuStckCnt" in df.columns:
        print(
            "발행주식수:",
            df["issuStckCnt"].iloc[0]
        )

    if "stckParPrc" in df.columns:
        print(
            "액면가:",
            df["stckParPrc"].iloc[0]
        )

## 3. 검증 결과

| 검증 항목 | 결과 |
|---|---|
| API 연결 | 성공 |
| 삼성전자 조회 | 성공 |
| ISIN 조회 | 성공 |
| 발행회사 정보 | 확인 가능 |
| 법인 식별정보 | 확인 가능 |
| 발행주식수 | 확인 가능 |
| 액면가 | 확인 가능 |
| 상장 관련 정보 | 확인 가능 |
| D06과 연결 가능한 식별정보 | 확인 |

### 확인 사항

- 삼성전자 단축 종목코드: `005930`
- 삼성전자 ISIN: `KR7005930003`
- D08은 단축 종목코드를 직접 주요 조회키로 사용하기보다 ISIN 등의 식별값을 활용한다.
- D06 주식시세정보에도 ISIN 필드가 존재하므로 D06과 D08을 ISIN 기준으로 연결할 수 있다.

### Timeout 관련

초기 테스트에서는 ISIN만 조회조건으로 사용했을 때 `ReadTimeout`이 발생했다.

조회조건을 다음과 같이 좁힌 후 정상 응답을 확인하였다.

- ISIN: `KR7005930003`
- 기준일자: `20261008`

따라서 실제 활용 시에도 지나치게 넓은 조건으로 조회하지 않도록 주의한다.